# MovieLens K-means クラスタリング
## ユーザーの映画嗜好グループ分析

**データ**: MovieLens ml-32m より抽出  
**手法**: K-means法  
**目的**: 映画の評価パターンからユーザーを嗜好グループに分類する


## 1. ライブラリのインポート

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
print("ライブラリの読み込み完了")


## 2. データの読み込み

In [ ]:
# CSVファイルの読み込み
df = pd.read_csv('movielens_50users.csv')

print(f"データ件数: {len(df):,}")
print(f"ユーザー数: {df['userId'].nunique()}")
print(f"映画数: {df['title'].nunique()}")
print()
df.head(10)


## 3. 基本統計・データ確認

In [ ]:
print("=== 評価の分布 ===")
print(df['rating'].describe())
print()
print("=== 映画一覧 ===")
for title in sorted(df['title'].unique()):
    count = df[df['title'] == title]['rating'].count()
    mean  = df[df['title'] == title]['rating'].mean()
    print(f"  {title[:50]:<50}  評価数:{count}  平均:{mean:.2f}")


## 4. 前処理

以下の手順で前処理を行う。

1. **ユーザー×映画の評価行列**を作成
2. **欠損値の補完**（各ユーザーの平均評価で補完）
3. **標準化**（StandardScaler）


In [ ]:
# ステップ1: ユーザー×映画の評価行列
user_movie = df.pivot_table(index='userId', columns='title', values='rating')
print(f"評価行列のサイズ: {user_movie.shape[0]} ユーザー × {user_movie.shape[1]} 映画")
print(f"欠損率: {user_movie.isnull().sum().sum() / user_movie.size:.1%}")
print()

# ステップ2: 欠損値をユーザー平均で補完
user_means = user_movie.mean(axis=1)
matrix_filled = user_movie.apply(lambda row: row.fillna(user_means[row.name]), axis=1)
print("欠損値補完後:")
print(f"  欠損数: {matrix_filled.isnull().sum().sum()}")

# ステップ3: 標準化
scaler = StandardScaler()
matrix_scaled = scaler.fit_transform(matrix_filled)
print(f"\n標準化後の形状: {matrix_scaled.shape}")
print("前処理完了")


## 5. 最適クラスタ数の決定

エルボー法とシルエットスコアで最適な k を選ぶ。


In [ ]:
k_range = range(2, 9)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(matrix_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(matrix_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method', fontsize=13)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, silhouettes, 'rs-', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score', fontsize=13)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = k_range.start + silhouettes.index(max(silhouettes))
print(f"シルエットスコア最大: k={best_k}  (score={max(silhouettes):.3f})")


## 6. K-means クラスタリングの実行

In [ ]:
K = 3  # エルボー法・シルエットスコアを参考に設定

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
labels = kmeans.fit_predict(matrix_scaled)

# ラベルを元のデータに付与
matrix_filled['cluster'] = labels

print(f"クラスタ数: {K}")
print()
print("クラスタ別ユーザー数:")
for i in range(K):
    print(f"  クラスタ {i}: {(labels == i).sum()} 人")


## 7. クラスタごとの特徴分析

In [ ]:
# 各クラスタの映画別平均評価
movie_cols = [c for c in matrix_filled.columns if c != 'cluster']
cluster_means = matrix_filled.groupby('cluster')[movie_cols].mean()

print("クラスタ別・映画別 平均評価スコア:")
print(cluster_means.round(2))
print()
print("各クラスタの好きな映画 Top3:")
for i in range(K):
    top3 = cluster_means.loc[i].nlargest(3)
    print(f"  クラスタ {i}: " + ", ".join([f"{m.split('(')[0].strip()}({v:.1f})" for m, v in top3.items()]))


## 8. 可視化①：クラスタ別評価ヒートマップ

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

# タイトルを短縮表示
short_titles = [t.split('(')[0].strip()[:25] for t in cluster_means.columns]
heatmap_data = cluster_means.copy()
heatmap_data.columns = short_titles

sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd',
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Avg Rating'})
ax.set_title('クラスタ別 映画評価ヒートマップ', fontsize=13, pad=10)
ax.set_xlabel('Movie Title')
ax.set_ylabel('Cluster')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. 可視化②：PCA による2次元散布図

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(matrix_scaled)

colors = ['#e94560', '#2eb872', '#f5a623']
fig, ax = plt.subplots(figsize=(8, 6))

for i in range(K):
    mask = labels == i
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=colors[i], s=80, alpha=0.8, label=f'Cluster {i}', edgecolors='white', linewidths=0.5)

ax.set_title('K-means クラスタリング結果（PCA 2次元）', fontsize=13)
ax.set_xlabel(f'PC1 (explained: {pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 (explained: {pca.explained_variance_ratio_[1]:.1%})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"2成分の累積寄与率: {pca.explained_variance_ratio_.sum():.1%}")


## 10. まとめ

| クラスタ | 人数 | 特徴 |
|---|---|---|
| Cluster 0 | - | 高評価傾向のユーザー層 |
| Cluster 1 | - | 中評価傾向のユーザー層 |
| Cluster 2 | - | 低評価傾向のユーザー層 |

K-means法により、映画の評価パターンに基づいてユーザーを嗜好グループに分類できた。  
各クラスタの特徴を把握することで、グループに合わせた映画レコメンドへの応用が期待できる。
